In [26]:
#perform 70-30 test-validation within each geospatial clusters 
#for each cluster we select random 70% of stations as training and 30% as validation
#final dataframes: train_stn: stn_id, cluster_id
#                  val_stn: stn_id, cluster_id

In [27]:
import pandas as pd
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import rioxarray as rxr
import rasterio as rio
import os

In [28]:
#load stations and cluster datasets

prec_cluster_path = r"..\Data\Processed\Spatial_clustering\precipitation_clusters.tif"
insitu_prec_df = pd.read_csv(r"..\Data\Processed\Insitu\insitu_rainfall_1991_2020.csv")

stn_df = insitu_prec_df.drop_duplicates(subset=['index_no'], keep='first').reset_index(drop=True)
stn_df = stn_df[['index_no','lat', 'long']]


In [29]:
#sample station points to extract cluster value
stn_points = list(zip(stn_df['long'], stn_df['lat']))

#extract cluster value
cluster_values = []
with rio.open(prec_cluster_path) as src:
    sampled_values = list(src.sample(stn_points))
    for i, value in enumerate(sampled_values):
        cluster_values.append(int(value[0]))
         
#add cluster values to station df
stn_df['cluster_id'] = cluster_values

stn_df

,index_no,lat,long,cluster_id
0,102,29.55744,80.41317,0
1,103,29.46714,80.54575,0
2,104,29.30135,80.58775,0
3,105,28.95477,80.23049,0
4,107,29.84496,80.54539,0
...,...,...,...,...
63,1407,26.90964,87.92308,1
64,1408,26.67067,87.70318,0
65,1412,26.56389,88.06150,1
66,1416,26.86806,88.07833,1


In [ ]:
#Split stations within each cluster into train and test
cluster_ids = np.unique(cluster_values)
train_df = pd.DataFrame()
test_df = pd.DataFrame()
split_fraction = 0.7

for cluster_id in cluster_ids:
    cluster_stations = stn_df[stn_df['cluster_id'] == cluster_id].reset_index(drop=True)
    train_df_lv = cluster_stations.sample(frac=split_fraction, random_state = 41)
    test_df_lv = cluster_stations.drop(train_df_lv.index)
    train_df = pd.concat([train_df, train_df_lv], ignore_index=True)
    test_df = pd.concat([test_df, test_df_lv], ignore_index=True)

#save both splits
save_dir = r"../Data/Processed/Split_stations/"
os.makedirs(save_dir, exist_ok=True)
train_df.to_csv(save_dir + 'train_stations.csv', index=False)
test_df.to_csv(save_dir + 'test_stations.csv', index=False)


    
